In [ ]:
import os

storage_account_name = os.getenv("AZURE_STORAGE_ACCOUNT_NAME")
storage_account_key = os.getenv("AZURE_STORAGE_ACCOUNT_KEY")
container_name = os.getenv("AZURE_CONTAINER_NAME", "redditta")

if not storage_account_name or not storage_account_key:
    raise ValueError("Thiếu biến môi trường AZURE_STORAGE_ACCOUNT_NAME hoặc AZURE_STORAGE_ACCOUNT_KEY")

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.blob.core.windows.net",
    storage_account_key
)

print("Cấu hình thành công")

In [0]:
display(dbutils.fs.ls("wasbs://reddit-data@chungminhlamkhaiphadl.blob.core.windows.net/"))

path,name,size,modificationTime
wasbs://reddit-data@chungminhlamkhaiphadl.blob.core.windows.net/.cache/,.cache/,0,0
wasbs://reddit-data@chungminhlamkhaiphadl.blob.core.windows.net/pushshift_1/,pushshift_1/,0,0


In [0]:

path = f"wasbs://{container_name}@{storage_account_name}.blob.core.windows.net/pushshift_1/*.parquet"

df = spark.read.parquet(path).select("author", "subreddit")

print(f"Tổng số dòng hiện có: {df.count()}")
display(df.limit(5))

Tổng số dòng hiện có: 1562841183


author,subreddit
Anaphase,196
TrollaBot,196
ToastPuppy15,196
AnarchoFuturist,196
Degenerate__filth,196


In [0]:
from pyspark.sql.functions import col, count

known_bots = [
    "AutoModerator", "RemindMeBot", "QualityVote", "Bot_Metric", 
    "haikusbot", "WikiTextBot", "vredditDownloader", "image_linker_bot",
    "TrollaBot", "bot", "Bot" 
]

# lọc bots
df_clean = df.filter(~col("author").isin(known_bots)) \
             .filter(~col("author").rlike("(?i)bot")) \
             .filter(col("author").isNotNull()) \
             .filter(col("subreddit").isNotNull())

df_distinct = df_clean.dropDuplicates(["author", "subreddit"])

# lọc user
user_counts = df_distinct.groupBy("author").agg(count("subreddit").alias("sub_count"))
valid_users = user_counts.filter(col("sub_count") < 50)

# join user
df_optimized = df_distinct.join(valid_users, "author", "inner").drop("sub_count")

# self-join
df_source = df_optimized.withColumnRenamed("subreddit", "source")
df_target = df_optimized.withColumnRenamed("subreddit", "target")

edges_raw = df_source.alias("a").join(df_target.alias("b"), "author") \
                     .filter(col("a.source") < col("b.target"))

#tính trọng số
edges_final = edges_raw.groupBy("source", "target").agg(count("author").alias("weight"))

# lọc nhiễu
edges_cleaned = edges_final.filter(col("weight") >= 5)

display(edges_cleaned.orderBy(col("weight").desc()).limit(20))

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:136)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:728)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:446)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:446)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

In [0]:
output_path = f"wasbs://{container_name}@{storage_account_name}.blob.core.windows.net/reddit_edges_result"
edges_cleaned.repartition(1).write.mode("overwrite").csv(output_path, header=True)

print("Đã lưu kết quả vào storage")

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:138)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:728)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:446)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:446)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:468)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:571)
	at com.data